In [1]:

import os 
os.chdir(os.path.dirname(os.getcwd()))


In [2]:
import ast
import pandas as pd
from sklearn.metrics import f1_score as f1 
from sklearn.metrics import precision_recall_fscore_support as prf

## Obitools

In [6]:


def can_family(tax_ids,database):

    sub = database[database['taxid_ncbi'].isin(tax_ids)]
    
    family = sub['family'].unique()

    if len(family) > 1:
        return 'NC_'
    return family[0]

acc = []
f1_scrores = []
p_score =[]
r_score =[]

OBI_pred_path = 'Obitools/scripts/results/obi3/Ac16'

for fold in range(1,7):

    print("-----------------")
    database = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/train.csv')

    preds = pd.read_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3.csv', sep="\t")



    preds['BEST_MATCH_IDS'] = preds['BEST_MATCH_IDS'].apply(ast.literal_eval)  # Convert to list of strings
    preds['BEST_MATCH_TAXIDS'] = preds['BEST_MATCH_TAXIDS'].apply(ast.literal_eval) 

    preds['family'] = preds['BEST_MATCH_TAXIDS'].apply(lambda x: can_family(x,database))

    labels = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/test.csv')

    classes = labels['family'].unique()

    preds['ak_family'] = labels['family']


    p,r,f,s = prf(preds["ak_family"], preds["family"], average="macro", labels=classes, zero_division=0)
    print(f'Fold {fold} f1: {f}')
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')

    
    # preds.to_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3_pred.csv', sep="\t", index=False)
    





-----------------
Fold 1 f1: 0.7391247320613276
Fold 1 precision: 0.7298561336564741
Fold 1 recall: 0.7935396261230436
-----------------
Fold 2 f1: 0.7440204659235242
Fold 2 precision: 0.7362337380212524
Fold 2 recall: 0.7882763473606497
-----------------
Fold 3 f1: 0.7535099040482591
Fold 3 precision: 0.7398445116468372
Fold 3 recall: 0.8033874179223017
-----------------
Fold 4 f1: 0.7621606974534593
Fold 4 precision: 0.753359152035291
Fold 4 recall: 0.8066602410149223
-----------------
Fold 5 f1: 0.7216303790996264
Fold 5 precision: 0.7126605388813204
Fold 5 recall: 0.7774918603933498
-----------------
Fold 6 f1: 0.742148862691689
Fold 6 precision: 0.728422623687297
Fold 6 recall: 0.7923500863907841


## DNABERT_2

In [3]:
padnas_macro= []
F = []
for fold in [1,2,3,4,5,6]:
    print("-----------------")
    preds = pd.read_csv(f'DNABert2/experiments/fine_tune_taxa/outputs/teleo_inference_no_cefe/checkpoints/fold_{fold}/predictions.csv')
    
    print("fold",fold)
    p,r,f,s = prf(preds['labels_family'],preds['preds_family'],average='macro', labels=preds['labels_family'].unique(), zero_division=0)
    print('p',p,'r',r,'f',f)

    

-----------------
fold 1
p 0.4642533434341877 r 0.55789263076207 f 0.4836824647697926
-----------------
fold 2
p 0.4481972977970539 r 0.5577167386831203 f 0.4658507572767918
-----------------
fold 3
p 0.4796223957146761 r 0.5675838102955517 f 0.4911660270706371
-----------------
fold 4
p 0.47466540951553826 r 0.5656663083742823 f 0.4893774999603046
-----------------
fold 5
p 0.45564516147997575 r 0.5616006830912543 f 0.47518455491297984
-----------------
fold 6
p 0.48667309766064204 r 0.5706910078846353 f 0.5022582543266843


## MMSeq2


In [5]:

for fold in [1,2,3,4,5,6]:

    preds_df = pd.read_csv(f'MMseqs2/result/Ac16/folds/fold_{fold}/tax_out/tax_out_lca.tsv', sep="\t", header=None)
    ground_truth = pd.read_csv(f'MMseqs2/Ac16/folds/fold_{fold}/test.csv')
    print("====================")
    print("fold",fold)

    preds_df[0] = preds_df[0].apply(lambda x: x.split('_')[-1]) # remove the prefix

    p,r,f,s = prf(preds_df[3],preds_df[0],average='macro',labels=ground_truth['family'].unique(),zero_division=0)
    print('p',p,'r',r,'f',f)

    


fold 1
p 0.7236272103553578 r 0.660360005310299 f 0.6696587424115384
fold 2
p 0.7296202284061382 r 0.6615737151791109 f 0.6696815668723053
fold 3
p 0.7288212971352506 r 0.6659262681807578 f 0.6750190543111333
fold 4
p 0.7270389157134247 r 0.6596251446771642 f 0.6740480846576873
fold 5
p 0.7106469111575895 r 0.6465934052858358 f 0.6547682525468923
fold 6
p 0.7326050510643534 r 0.6604791966859069 f 0.6758773585135801
